# Extending Acquisition Functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/extending_base_classes/acquisition_functions.ipynb)

_Open in Colab works once ALF is public / on PyPI; until then, use the local setup below._

This tutorial shows how to create custom acquisition functions by extending [`AcquisitionFunction`](https://instadeepai.github.io/alf/api/alf_core/optimizer/acquisition_function/). We'll implement uncertainty sampling and diversity-based acquisition as examples.

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
%pip install "git+https://github.com/instadeepai/alf.git#subdirectory=core"
# Once ALF is on PyPI this simplifies to `%pip install alf_core` (no git URL).

## 1. Imports

In [ ]:
import numpy as np
from alf_core.dataclasses import Candidate, LabelledCandidates, Modality, Predictions, State
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.model.base_model import BaseModel
from alf_core.optimizer.acquisition_function import AcquisitionFunction
from alf_core.surrogate.surrogate import Surrogate
from alf_core.utils.enums import ProblemType

## 2. Define Custom Acquisition Functions

Implement the `__call__()` method to score candidates based on your acquisition strategy. The method receives a list of [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects and a [`State`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/state/) containing the surrogate model and dataset. It must return [`LabelledCandidates`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/labelled_candidates/) where labels represent acquisition scores.

**Uncertainty sampling** scores each candidate by its prediction variance — pure exploration. `__call__` returns `LabelledCandidates` whose labels are the acquisition scores (higher = selected first).

In [ ]:
class UncertaintySampling(AcquisitionFunction):
    """Select candidates with highest prediction uncertainty."""

    def __call__(
        self,
        search_candidates: list[Candidate],
        state: State,
    ) -> LabelledCandidates:
        """Score candidates by their prediction variance."""
        # Get predictions from surrogate model
        predictions = state.surrogate.predict(search_candidates)

        # Use variance as acquisition score (higher = more uncertain)
        if predictions.variances is not None:
            scores = predictions.variances
        else:
            # Fallback if no variance available
            scores = np.ones(len(search_candidates))

        return LabelledCandidates(candidates=search_candidates, labels=scores)

**Upper Confidence Bound (UCB)** adds the mean and a scaled standard deviation (`mean + β·std`), trading off exploitation against exploration via `β`.

In [ ]:
class UpperConfidenceBound(AcquisitionFunction):
    """UCB acquisition: balance exploitation and exploration.

    β (beta) is the exploration parameter that controls how much uncertainty
    is weighted when selecting actions. It scales the confidence interval term
    added to the estimated value, balancing exploration vs. exploitation:
    higher β → more exploration; lower β → more exploitation.
    """

    def __init__(self, beta: float = 2.0):
        self.beta = beta

    def __call__(
        self,
        search_candidates: list[Candidate],
        state: State,
    ) -> LabelledCandidates:
        """Score candidates using UCB: mean + beta * std."""
        predictions = state.surrogate.predict(search_candidates)

        # UCB formula
        means = predictions.means
        if predictions.variances is not None:
            stds = np.sqrt(predictions.variances)
            scores = means + self.beta * stds
        else:
            scores = means

        return LabelledCandidates(candidates=search_candidates, labels=scores)

**Diversity sampling** ignores the surrogate entirely and scores each candidate by its distance to the nearest training point, favouring under-explored regions.

In [ ]:
class DiversitySampling(AcquisitionFunction):
    """Select diverse candidates based on distance from training data."""

    def __call__(
        self,
        search_candidates: list[Candidate],
        state: State,
    ) -> LabelledCandidates:
        """Score candidates by minimum distance to training set."""
        # Extract candidate data
        candidate_data = np.array([c.data for c in search_candidates])
        training_data = np.array([c.data for c in state.dataset.train_dataset.candidates])

        # Compute minimum distance to any training point
        scores = []
        for point in candidate_data:
            distances = np.abs(training_data - point)
            min_distance = np.min(distances) if len(distances) > 0 else 1.0
            scores.append(min_distance)

        scores = np.array(scores)
        return LabelledCandidates(candidates=search_candidates, labels=scores)

## 3. Usage Example

To exercise these, we build a `State` with a toy surrogate (returns random means/variances) and a small training split for the diversity score to measure distance from.

In [ ]:
# Acquisition functions receive the full task `State`. We build one with:
#  - a surrogate that can predict (a toy model wrapped in `Surrogate`)
#  - a dataset whose training split the diversity score measures distance from
#    (see the "Extending Datasets" tutorial for the dataset pattern)
class InMemoryDataset(BaseDataset):
    """Minimal dataset backed by candidates and labels already in memory."""

    def __init__(self, candidates, labels, config):
        super().__init__(config)
        self._candidates = candidates
        self._labels = labels

    def load_dataset(self) -> LabelledCandidates:
        return LabelledCandidates(candidates=self._candidates, labels=self._labels)


class MockModel(BaseModel):
    """Toy surrogate that returns random means and variances."""

    def featurise(self, inputs):
        return np.array([c.data for c in inputs]).reshape(-1, 1)

    def train(self, train_data, val_data):
        pass

    def predict(self, candidate_points):
        n = len(candidate_points)
        return Predictions(means=np.random.rand(n), variances=np.random.rand(n) * 0.1)

    def sample(self, condition=None):
        return []

Assemble the dataset and `State`:

In [ ]:
# Some training data; the diversity score measures distance from the training split
train_xs = np.linspace(0.0, 9.0, 10)
training_candidates = [Candidate(data=float(x), modality=Modality.TABULAR) for x in train_xs]
training_labels = np.array([float(x) for x in train_xs])
config = BaseDatasetConfig(
    name="demo",
    modality=Modality.TABULAR,
    seed=0,
    train_ratio=0.6,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type="random",
    problem_type=ProblemType.REGRESSION,
)
dataset = InMemoryDataset(training_candidates, training_labels, config)
dataset.setup()

state = State(
    dataset=dataset,
    surrogate=Surrogate(model=MockModel()),
    round=1,
    acq_batch_size=5,
)

Generate a pool of search candidates and score them with each strategy, taking the top 5 via `get_top_k`:

In [ ]:
# Generate search candidates
search_candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in np.linspace(0, 10, 20)]

# Apply different acquisition functions
print("=== Uncertainty Sampling ===")
uncertainty_acq = UncertaintySampling()
scored = uncertainty_acq(search_candidates, state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")
print(f"Top 5 candidates: {[c.data for c in top_5.candidates]}")

print("\n=== Upper Confidence Bound ===")
ucb_acq = UpperConfidenceBound(beta=2.0)
scored = ucb_acq(search_candidates, state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")

print("\n=== Diversity Sampling ===")
diversity_acq = DiversitySampling()
scored = diversity_acq(search_candidates, state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")
print(f"Top 5 candidates: {[c.data for c in top_5.candidates]}")

## Key Points

- **Required method**: `__call__(search_candidates, state)` must return `LabelledCandidates`
- **Input**: Receives unlabelled candidates from search function and current task state
- **Output**: Return candidates with acquisition scores as labels (higher = better)
- **Surrogate access**: Use `state.surrogate.predict()` to get predictions
- **Training data**: Access via `state.dataset` to avoid re-selecting points
- **Scoring**: Higher scores indicate candidates worth evaluating
- **Selection**: Framework automatically selects top-k candidates based on scores

Common acquisition strategies:
- **Greedy**: Select highest predicted value (exploitation)
- **Uncertainty**: Select most uncertain predictions (pure exploration)
- **UCB**: Balance exploitation and exploration with beta parameter
- **Expected Improvement**: Probability of improvement over current best
- **Thompson Sampling**: Sample from posterior distribution
- **Diversity**: Maximize distance from existing points

See existing implementations in `tools/alf_tools/optimizer/acquisition_functions/` for more examples.